웹 서버 로그 기반 악성 요청 탐지를 위한 분류 모델 구축

In [25]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# 1. 데이터 로드 
df = pd.read_csv('web_server_logs.csv')

# 2. 전처리: hour 추출
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour

# 3. 전처리: status_code -> is_error, label 생성
df['is_error'] = np.where(df['status_code'] >= 400, 1, 0)
df['label'] = df['is_error']

# 4. 전처리: method 원핫인코딩
df = pd.get_dummies(df, columns=['method'], dtype=int)

# 5. 전처리: size -> 로그 변환
df['size'] = np.log1p(df['size'])

# 6. 데이터 분리
X = df.drop(columns=['timestamp', 'status_code', 'is_error', 'label','ip'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # stratify 추가 권장

# 7. 모델 학습 및 예측
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

# 임계값 조정
target_probabilities = model.predict_proba(X_test)[:, 1]

custom_threshold = 0.45
y_pred_new = (target_probabilities >= custom_threshold).astype(int)

# 8. 성능 평가 및 출력
print("=== 분류 모델 평가 결과 ===")
print("정확도 (Accuracy) :", accuracy_score(y_test, y_pred_new))
print("정밀도 (Precision):", precision_score(y_test, y_pred_new))
print("재현율 (Recall)   :", recall_score(y_test, y_pred_new))
print("F1-Score          :", f1_score(y_test, y_pred_new))
print("\n=== 상세 리포트 ===")
print(classification_report(y_test, y_pred_new))

"""
======================================================================
[최종 모델 해석 및 요약 보고서] - 작성자: 정병규

1. 기본 모델 (Threshold=0.5) 결과 분석:
   - Accuracy: ~0.50, Recall: ~0.55, F1-Score: ~0.39
   - 데이터 불균형(정상 213건 vs 에러 87건) 상황에서 class_weight='balanced'를 통해 
     최소한의 악성 탐지력을 확보함.

2. 임계값 조정 실험 (Threshold=0.45) 결과 및 한계점:
   - 임계값을 0.45로 낮추었을 때 악성 요청 Recall이 93.1%까지 대폭 상승하고 
     종합 지표인 F1-Score도 0.438로 향상됨.
   - 그러나 정상 샘플(0)의 Recall이 5%로 급감하며 대부분의 정상 로그를 악성으로 오탐하는 
     과잉 탐지(False Positive) 현상이 심화됨.

3. 결론 및 제언:
   - 보안 관제 특성상 미탐을 줄이는 Recall 93% 모델이 이론적으로 우수해 보일 수 있으나, 
     현재 로지스틱 회귀 모델로는 정상과 악성의 경계를 깔끔하게 분리하는 데 한계가 있음.
   - 따라서 현업 적용을 위해서는 무리한 임계값 조정보다는 
     Random Forest, XGBoost 등 비선형 패턴을 학습할 수 있는 앙상블 모델로의 고도화가 필수적임.
======================================================================
"""

=== 분류 모델 평가 결과 ===
정확도 (Accuracy) : 0.30666666666666664
정밀도 (Precision): 0.2862190812720848
재현율 (Recall)   : 0.9310344827586207
F1-Score          : 0.43783783783783786

=== 상세 리포트 ===
              precision    recall  f1-score   support

           0       0.65      0.05      0.10       213
           1       0.29      0.93      0.44        87

    accuracy                           0.31       300
   macro avg       0.47      0.49      0.27       300
weighted avg       0.54      0.31      0.19       300



"\n======================================================================\n[최종 모델 해석 및 요약 보고서] - 작성자: 정병규\n\n1. 기본 모델 (Threshold=0.5) 결과 분석:\n   - Accuracy: ~0.50, Recall: ~0.55, F1-Score: ~0.39\n   - 데이터 불균형(정상 213건 vs 에러 87건) 상황에서 class_weight='balanced'를 통해 \n     최소한의 악성 탐지력을 확보함.\n\n2. 임계값 조정 실험 (Threshold=0.45) 결과 및 한계점:\n   - 임계값을 0.45로 낮추었을 때 악성 요청 Recall이 93.1%까지 대폭 상승하고 \n     종합 지표인 F1-Score도 0.438로 향상됨.\n   - 그러나 정상 샘플(0)의 Recall이 5%로 급감하며 대부분의 정상 로그를 악성으로 오탐하는 \n     과잉 탐지(False Positive) 현상이 심화됨.\n\n3. 결론 및 제언:\n   - 보안 관제 특성상 미탐을 줄이는 Recall 93% 모델이 이론적으로 우수해 보일 수 있으나, \n     현재 로지스틱 회귀 모델로는 정상과 악성의 경계를 깔끔하게 분리하는 데 한계가 있음.\n   - 따라서 현업 적용을 위해서는 무리한 임계값 조정보다는 \n     Random Forest, XGBoost 등 비선형 패턴을 학습할 수 있는 앙상블 모델로의 고도화가 필수적임.\n======================================================================\n"